In [1]:
from basic import TasnsfModel
from tokenizer.tokenizer import encode, decode
import torch
import time
device = "cuda:0"

In [2]:
model = TasnsfModel(vocab_size=25000, max_seq_len=1024, embd_dim=2048, num_head=16, num_layers=20).to(device)

In [ ]:
model.load_state_dict(torch.load("ft_models/epoch_17.pth", map_location=device))
model.eval()

TasnsfModel(
  (embedding): Embedding(25000, 2048)
  (pos_embd): Embedding(1024, 2048)
  (transf_blocks): ModuleList(
    (0-19): 20 x TransfBlock(
      (multi_head_attn): MultiHeadAttn(
        (heads): ModuleList(
          (0-15): 16 x SelfAttn(
            (Q): Linear(in_features=2048, out_features=128, bias=False)
            (K): Linear(in_features=2048, out_features=128, bias=False)
            (V): Linear(in_features=2048, out_features=128, bias=False)
            (attn_dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (out_proj): Linear(in_features=2048, out_features=2048, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (dropout): Dropout(p=0.2, inplace=False)
      (ln1): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (ff1): Linear(in_features=2048, out_features=8192, bias=True)
      (ff2): Linear(in_features=8192, out_features=2048, bias=True)
  

In [4]:
def apply_repet_penalty(logits, gen_tokens, repet_penalty):
    for token in set(gen_tokens):
        logit = logits[token]
        if logit > 0:
            logits[token] = logit / repet_penalty
        else:
            logits[token] = logit * repet_penalty
            
    return logits

In [5]:
def top_p_filter(probs, p):
    sorted_probs, indices = torch.sort(probs, descending=True)
    cum_probs = torch.cumsum(sorted_probs, -1)
    
    cutoff_mask = cum_probs > p
    cutoff_mask[0] = False
    sorted_probs[cutoff_mask] == 0.0
    
    filtered_probs = torch.zeros_like(probs)
    filtered_probs.scatter_(0, indices, sorted_probs)
    
    filtered_probs = filtered_probs / filtered_probs.sum()
    
    return filtered_probs

In [11]:
def generate(text, max_gen_len=128, temp=0.3, top_p=0.8, repet_penalty=1.2):

    inp_tokens = encode(text)
    gen_tokens = []
    next_token = -1
    eos_token = encode("<eos>")[0]

    while next_token != eos_token and len(gen_tokens) < max_gen_len:
        tokens = inp_tokens + gen_tokens
        tokens = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
        out = model(tokens)
        
        logits = out[:, -1]
        logits = logits.squeeze(0)
        logits = apply_repet_penalty(logits, gen_tokens, repet_penalty)
        logits = logits / temp
        
        probs = torch.softmax(logits, dim=-1)
        probs = top_p_filter(probs, p=top_p)
        next_token = torch.multinomial(probs, 1).item()
        
        gen_tokens.append(next_token)
        try:
            print(decode([next_token]), end="")
        except:
            pass

    return decode(gen_tokens)

In [13]:
question = '''meaning of "dharma"?'''

prompt = "<user>" + question + "<system>"
print(prompt)

op = generate(prompt)
# print("\n\n", "- "*20, "\n\n", op)

<user>meaning of "dharma"?<system>
the term "dharma" refers to a specific duty or office that is considered righteous. it can refer to the duties of a king, a king, and also to the duties of an ascetic (stayed in pious practices).<eos>